# 4. Design and implement a CNN for Image Classification a) Select a suitable image classification dataset (medical imaging, agricultural, etc.). b) Optimized with different hyper-parameters including learning rate, filter size, no. of layers, optimizers, dropouts, etc.

# Download Dataset from Kaggle link : https://www.kaggle.com/datasets/ayanwap7/rice-image-dataset-train-test-split/data

In [ ]:
import os, shutil
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix

# Basic setup
img_size = (150, 150)
batch_size = 32
base_dir = "Rice_dataset/Rice_Image_Dataset"
train_dir = os.path.join(base_dir, "train")
test_dir = os.path.join(base_dir, "test")
val_dir = os.path.join(base_dir, "validation")

# Split validation from training (only once)
if not os.path.exists(val_dir):
    os.makedirs(val_dir)
    for cls in os.listdir(train_dir):
        cls_path = os.path.join(train_dir, cls)
        files = os.listdir(cls_path)
        np.random.shuffle(files)
        split = int(0.2 * len(files))
        os.makedirs(os.path.join(val_dir, cls), exist_ok=True)
        for f in files[:split]:
            shutil.move(os.path.join(cls_path, f), os.path.join(val_dir, cls, f))

# Data generators
gen = ImageDataGenerator(rescale=1./255)
train_data = gen.flow_from_directory(train_dir, target_size=img_size, batch_size=batch_size, class_mode='categorical')
val_data = gen.flow_from_directory(val_dir, target_size=img_size, batch_size=batch_size, class_mode='categorical')
test_data = gen.flow_from_directory(test_dir, target_size=img_size, batch_size=1, class_mode='categorical', shuffle=False)

# CNN Model
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(150,150,3)),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(train_data.num_classes, activation='softmax')
])

model.compile(optimizer=Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])

# Train
model.fit(train_data, epochs=1, validation_data=val_data)

# Evaluate
preds = model.predict(test_data)
y_true = test_data.classes
y_pred = np.argmax(preds, axis=1)
labels = list(test_data.class_indices.keys())

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, cmap='Blues')
plt.xlabel("Predicted"), plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

# Report
print("Classification Report:\n", classification_report(y_true, y_pred, target_names=labels))


Found 48000 images belonging to 5 classes.
Found 12000 images belonging to 5 classes.
Found 15000 images belonging to 5 classes.
 506/1500 [=========>....................] - ETA: 3:47 - loss: 0.2617 - accuracy: 0.9077